In [1]:
import datetime
from collections import defaultdict

import numpy as np
import earthkit.data as ekd
# import earthkit.regrid as ekr

from anemoi.inference.runners.simple import SimpleRunner
from anemoi.inference.outputs.printer import print_state

from ecmwf.opendata import Client as OpendataClient
import time

from scipy.sparse import load_npz
import logging
import pickle
import xarray as xr
import pandas as pd
import os

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

In [2]:
CHECKPOINT = "weights/AIFS/aifs-single-mse-1.0.ckpt"
LATLON_N320_PATH = "support/regrid/mir_16_linear/9533e90f8433424400ab53c7fafc87ba1a04453093311c0b5bd0b35fedc1fb83.npz"
TFM_LATLON_N320 = load_npz(LATLON_N320_PATH)
N320_LATLON_PATH = "support/regrid/mir_16_linear/7f0be51c7c1f522592c7639e0d3f95bcbff8a044292aa281c1e73b842736d9bf.npz"
TFM_N320_LATLON = load_npz(N320_LATLON_PATH)
ERA5_PATH = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
FULL_ERA5 = xr.open_zarr(ERA5_PATH, chunks=None)
LATITUDES = np.linspace(90, -90, 721)
LONGITUDES = np.linspace(0, 359.75, 1440)

INPUT_STATE_PATH = "input_states"
OUTPUT_STATE_PATH = "output_states"

for path in [INPUT_STATE_PATH, OUTPUT_STATE_PATH]:
    if not os.path.exists(path):
        os.makedirs(path)

In [3]:
def get_latest_IFS_data():
    logging.info("Using the latest IFS data from ECMWF OpenData")
    IFS_PARAM_SFC = [
        "10u",
        "10v",
        "2d",
        "2t",
        "msl",
        "skt",
        "sp",
        "tcw",
        "lsm",
        "z",
        "slor",
        "sdor",
    ]
    IFS_PARAM_SOIL = ["vsw", "sot"]
    IFS_PARAM_PL = ["gh", "t", "u", "v", "w", "q"]
    IFS_LEVELS = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50]
    IFS_SOIL_LEVELS = [1, 2]

    DATE = OpendataClient().latest()
    logging.info(f"Initial date is {DATE}")
    save_path = f"{INPUT_STATE_PATH}/input_state_{DATE.strftime('%Y%m%dT%H')}_IFS.pkl"
    if os.path.exists(save_path):
        logging.info(f"Input state for {DATE} already exists. Loading from file.")
        with open(save_path, "rb") as f:
            input_state = pickle.load(f)
        return input_state

    def get_open_data(param, levelist=[]):
        fields = defaultdict(list)
        # Get the data for the current date and the previous date
        for date in [DATE - datetime.timedelta(hours=6), DATE]:
            data = ekd.from_source(
                "ecmwf-open-data", date=date, param=param, levelist=levelist
            )
            for f in data:
                # Open data is between -180 and 180, we need to shift it to 0-360
                assert f.to_numpy().shape == (721, 1440)
                values = np.roll(f.to_numpy(), -f.shape[1] // 2, axis=1)
                # Interpolate the data to from 0.25 to N320
                # values = ekr.interpolate(
                #     values, {"grid": (0.25, 0.25)}, {"grid": "N320"}
                # )
                values = TFM_LATLON_N320 * values
                # Add the values to the list
                name = (
                    f"{f.metadata('param')}_{f.metadata('levelist')}"
                    if levelist
                    else f.metadata("param")
                )
                fields[name].append(values)

        # Create a single matrix for each parameter
        for param, values in fields.items():
            fields[param] = np.stack(values)

        return fields

    # Create empty fields dictionary
    fields = {}
    # Get the surface parameters
    fields.update(get_open_data(param=IFS_PARAM_SFC))
    # Get the soil parameters
    soil = get_open_data(param=IFS_PARAM_SOIL, levelist=IFS_SOIL_LEVELS)

    # Map the soil parameters to the expected names
    mapping = {"sot_1": "stl1", "sot_2": "stl2", "vsw_1": "swvl1", "vsw_2": "swvl2"}
    for k, v in soil.items():
        fields[mapping[k]] = v

    # Get the pressure level parameters
    fields.update(get_open_data(param=IFS_PARAM_PL, levelist=IFS_LEVELS))

    # Transform GH to Z
    for level in IFS_LEVELS:
        gh = fields.pop(f"gh_{level}")
        fields[f"z_{level}"] = gh * 9.80665

    input_state = dict(date=DATE, fields=fields)
    
    # write out the input state to a file with the date in the filename
    with open(save_path, "wb") as f:
        pickle.dump(input_state, f)

    return input_state


In [4]:
def get_ERA5(init_date):
    PARAM_PL_ERA5 = [
        "geopotential",
        "temperature",
        "u_component_of_wind",
        "v_component_of_wind",
        "vertical_velocity",
        "specific_humidity",
    ]
    PARAM_SFC_ERA5 = [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "2m_temperature",
        "2m_dewpoint_temperature",
        "mean_sea_level_pressure",
        "skin_temperature",
        "surface_pressure",
        "total_column_water",
        "land_sea_mask",
        "geopotential_at_surface",
        "sea_surface_temperature",
        "volumetric_soil_water_layer_1",
        "volumetric_soil_water_layer_2",
        "soil_temperature_level_1",
        "soil_temperature_level_2",
        "standard_deviation_of_orography",
        "slope_of_sub_gridscale_orography",
    ]
    RENAME_SFC = {
        "10m_u_component_of_wind": "10u",
        "10m_v_component_of_wind": "10v",
        "2m_temperature": "2t",
        "2m_dewpoint_temperature": "2d",
        "mean_sea_level_pressure": "msl",
        "skin_temperature": "skt",
        "surface_pressure": "sp",
        "total_column_water": "tcw",
        "land_sea_mask": "lsm",
        "geopotential_at_surface": "z",
        "sea_surface_temperature": "sst",
        "volumetric_soil_water_layer_1": "swvl1",
        "volumetric_soil_water_layer_2": "swvl2",
        "soil_temperature_level_1": "stl1",
        "soil_temperature_level_2": "stl2",
        "standard_deviation_of_orography": "sdor",
        "slope_of_sub_gridscale_orography": "slor",
    }
    RENAME_PL = {
        "geopotential": "z",
        "temperature": "t",
        "u_component_of_wind": "u",
        "v_component_of_wind": "v",
        "vertical_velocity": "w",
        "specific_humidity": "q",
    }
    LEVELS = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50]
    PARAM_SFC = [
        "10u",
        "10v",
        "2d",
        "2t",
        "msl",
        "skt",
        "sp",
        "tcw",
        "lsm",
        "z",
        "slor",
        "sdor",
        "stl1",
        "stl2",
        "swvl1",
        "swvl2",
    ]
    PARAM_PL = ["z", "t", "u", "v", "w", "q"]

    logging.info(f"Getting ERA5 data for date {init_date}")
    save_path = f"{INPUT_STATE_PATH}/input_state_{init_date.strftime('%Y%m%dT%H')}_ERA5.pkl"
    if os.path.exists(save_path):
        logging.info(f"Input state for {init_date} already exists. Loading from file.")
        with open(save_path, "rb") as f:
            input_state = pickle.load(f)
        return input_state
    init_date_minus_6 = init_date - datetime.timedelta(hours=6)

    logging.info("Getting pressure level data...")
    pl_ds = (
        FULL_ERA5[PARAM_PL_ERA5]
        .sel(time=[init_date_minus_6, init_date], level=LEVELS)
        .compute()
        .rename(RENAME_PL)
    )

    logging.info("Getting surface level data...")
    sfc_ds = (
        FULL_ERA5[PARAM_SFC_ERA5]
        .sel(time=[init_date_minus_6, init_date])
        .compute()
        .rename(RENAME_SFC)
    )

    logging.info("Processing surface level data...")
    fields_sfc = defaultdict(list)
    for date in sfc_ds.time:
        sfc_ds_date = sfc_ds.sel(time=date)
        for param in PARAM_SFC:
            values = sfc_ds_date[param].to_numpy().flatten()
            # print(values.shape)
            # values = ekr.interpolate(values, {"grid": (0.25, 0.25)}, {"grid": "N320"})
            values = TFM_LATLON_N320 * values
            # print(values.shape)
            fields_sfc[param].append(values)

    logging.info("Processing pressure level data...")
    fields_pl = defaultdict(list)
    for date in pl_ds.time:
        pl_ds_date = pl_ds.sel(time=date)
        for param in PARAM_PL:
            for level in LEVELS:
                values = pl_ds_date[param].sel(level=level).to_numpy().flatten()
                # print(values.shape)
                # values = ekr.interpolate(
                #     values, {"grid": (0.25, 0.25)}, {"grid": "N320"}
                # )
                values = TFM_LATLON_N320 * values
                # print(values.shape)
                fields_pl[f"{param}_{level}"].append(values)

    logging.info("Making input state...")
    fields = {}
    fields.update(fields_sfc)
    fields.update(fields_pl)

    for param, values in fields.items():
        fields[param] = np.stack(values)

    input_state = dict(date=init_date, fields=fields)

    logging.info("Saving input state to file...")
    with open(save_path, "wb") as f:
        pickle.dump(input_state, f)
        logging.info(f"Input state saved to {save_path}")

    logging.info(f"Input state for {init_date} created successfully.")
    return input_state

In [5]:
def process_step(output_state, runcount):
    data_vars = {}
    logging.info(f"Processing step {runcount}")
    for field in output_state['fields']:
        values = (TFM_N320_LATLON * output_state['fields'][field].reshape(-1,1)).reshape(721,1440)
        data_vars[field] = (["lat", "lon"], values.astype(np.float32))

    step_ds = xr.Dataset(
        data_vars,
        coords={"lat": LATITUDES, "lon": LONGITUDES},
    )
    step_ds = step_ds.expand_dims('step')
    step_ds['step'] = [int(runcount)]
    return step_ds

In [6]:
def run_inference(init_date=None, lead_time=360, save_vars=None):

    if lead_time < 6 or lead_time % 6 != 0:
        raise ValueError("Lead time must be a multiple of 6 hours and at least 6 hours.")

    if save_vars is None and lead_time > 120:
        logging.warning("Running this model for more than 120 steps and saving all variables is not recommended.")

    ic_src = "IFS" if init_date is None else "ERA5"

    # NEW: encode which vars are saved so different runs don't overwrite each other
    vars_tag = "ALL" if save_vars is None else "-".join(save_vars)
    save_path = f"{OUTPUT_STATE_PATH}/init_{ic_src}_{init_date.strftime('%Y%m%dT%H')}_lead_{lead_time}_vars_{vars_tag}.zarr"

    if os.path.exists(save_path):
        logging.info(f"Output file {save_path} already exists. Skipping inference.")
        logging.info("Loading existing dataset...")
        return xr.open_zarr(save_path)

    # build input state
    if ic_src == "IFS":
        input_state = get_latest_IFS_data()
    else:
        input_state = get_ERA5(init_date)

    runner = SimpleRunner(CHECKPOINT, device="cuda")
    current_step = 0
    start_time = time.perf_counter()
    print("Starting the inference session...")
    current_step_time = time.perf_counter()
    states = []
    for state in runner.run(input_state=input_state, lead_time=lead_time):
        print_state(state)
        current_step += 6
        if save_vars is None:
            processed_state = process_step(state, current_step)
        else:
            selected_data = {
                'date': state['date'],
                'fields': {var: state['fields'][var] for var in save_vars},
                'latitudes': state['latitudes'],
                'longitudes': state['longitudes'],
            }
            processed_state = process_step(selected_data, current_step)
        states.append(processed_state)
        logging.info(f"Step {current_step} completed.")
        step_time = time.perf_counter()
        logging.info(f"Time taken for step {current_step}: {step_time - current_step_time:.2f} s.")
        current_step_time = step_time

    logging.info("Inference session completed.")
    logging.info(f"Total time: {time.perf_counter() - start_time:.2f} s.")

    logging.info("Concatenating all steps into a single dataset.")
    ds = xr.concat(states, dim='step')
    del states
    ds = ds.expand_dims("time")
    ds["time"] = [pd.to_datetime(input_state['date'])]

    logging.info(f"Saving output dataset to {save_path}")
    ds.to_zarr(save_path, mode='w')

    return ds


In [7]:
import datetime

init_date = datetime.datetime(2023, 6, 1, 0, 0)
lead_time = 360
save_vars = ["2t", "tp", "z_500"]   # <-- add z_500

output_ds = run_inference(init_date=init_date, lead_time=lead_time, save_vars=save_vars)
print("variables in output_ds:", list(output_ds.data_vars))


2025-09-17 01:02:12,881 - INFO - Output file output_states/init_ERA5_20230601T00_lead_360_vars_2t-tp-z_500.zarr already exists. Skipping inference.
2025-09-17 01:02:12,882 - INFO - Loading existing dataset...


variables in output_ds: ['2t', 'tp', 'z_500']


In [20]:
%pip install ipympl



Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

import cartopy.crs as ccrs
import cartopy.feature as cfeature

VAR_MAP = {
    "2t":    "2m_temperature",      # Kelvin
    "z_500": "geopotential",        # m^2 s^-2 (convert to meters)
    "tp":    "total_precipitation", # meters (convert to mm)
}
G = 9.80665  # m s^-2

def _to_celsius(da: xr.DataArray) -> xr.DataArray:
    return (da - 273.15).assign_attrs(units="°C")

def _era5_align_to_aifs(da: xr.DataArray, aifs_like: xr.DataArray) -> xr.DataArray:
    ren = {}
    if "latitude" in da.dims:  ren["latitude"]  = "lat"
    if "longitude" in da.dims: ren["longitude"] = "lon"
    if ren: da = da.rename(ren)
    if float(da.lon.min()) < 0 or float(da.lon.max()) <= 180:
        da = da.assign_coords(lon=(da.lon % 360))
    if not np.all(np.diff(da.lon.values) > 0): da = da.sortby("lon")
    if not np.all(np.diff(da.lat.values) > 0): da = da.sortby("lat")
    out = da.interp(lat=aifs_like.lat.values, lon=aifs_like.lon.values)
    return out.sortby(["lat","lon"])

def make_triptych_widgets_static_refresh(
    output_ds: xr.Dataset,
    era5: xr.Dataset,
    aifs_var: str = "2t",
    vname_map: dict = VAR_MAP,
    use_percentile_limits: bool = True,
    diff_clip_pct: float = 99.5,
    add_coastlines=("targets","predictions"),  # <- which panels get coastlines
):
    if aifs_var not in output_ds:
        raise KeyError(f'"{aifs_var}" not in output_ds. Include it in save_vars before running inference.')
    if aifs_var not in vname_map:
        raise KeyError(f'No ERA5 mapping for "{aifs_var}". Add it to VAR_MAP.')

    init_time = pd.to_datetime(output_ds.time.values[0])
    steps = output_ds.step.values.astype(int)  # 6,12,...,360
    valid_times = [init_time + pd.to_timedelta(int(h), "h") for h in steps]

    pred = output_ds[aifs_var].isel(time=0).sortby(["lat","lon"])  # (step, lat, lon)

    era5_var = vname_map[aifs_var]
    if aifs_var == "z_500":
        tgt_raw = era5[era5_var].sel(time=valid_times, level=500, method="nearest") / G
        tgt_raw = tgt_raw.rename("z_500")
    else:
        tgt_raw = era5[era5_var].sel(time=valid_times, method="nearest")
    tgt = _era5_align_to_aifs(tgt_raw, pred).assign_coords(step=("time", steps)).swap_dims(time="step")

    # conversions
    if aifs_var == "2t":
        pred_plot = _to_celsius(pred);  tgt_plot = _to_celsius(tgt);  units = "°C"
    elif aifs_var == "z_500":
        pred_plot = (pred / G).assign_attrs(units="m");  tgt_plot = tgt.assign_attrs(units="m");  units = "m"
    elif aifs_var == "tp":
        pred_plot = (pred * 1000.0).assign_attrs(units="mm");  tgt_plot = (tgt * 1000.0).assign_attrs(units="mm");  units = "mm"
    else:
        pred_plot = pred; tgt_plot = tgt; units = pred.attrs.get("units","")

    diff = (pred_plot - tgt_plot).astype(np.float32)

    # absolute-panel limits
    if use_percentile_limits:
        pair = np.concatenate([pred_plot.values.ravel(), tgt_plot.values.ravel()])
        vmin_main = float(np.nanpercentile(pair, 0.5))
        vmax_main = float(np.nanpercentile(pair, 99.5))
    else:
        vmin_main = float(min(pred_plot.min().item(), tgt_plot.min().item()))
        vmax_main = float(max(pred_plot.max().item(), tgt_plot.max().item()))

    # diff limits (global)
    a_global = float(np.nanpercentile(np.abs(diff.values.ravel()), diff_clip_pct)) if use_percentile_limits \
               else float(np.nanmax(np.abs(diff.values)))

    extent = [float(pred_plot.lon.min()), float(pred_plot.lon.max()),
              float(pred_plot.lat.min()), float(pred_plot.lat.max())]
    out = widgets.Output()

    def nice_name(var):
        return {"2t":"2m_temperature", "z_500":"z500 (m)", "tp":"total_precipitation (mm)"} \
               .get(var, var)

    def draw(i: int):
        stp = int(steps[i])
        vmin_diff, vmax_diff = -a_global, a_global

        # Cartopy GeoAxes for coastlines
        proj = ccrs.PlateCarree()
        fig, axes = plt.subplots(
            1, 3, figsize=(18, 4), constrained_layout=True,
            subplot_kw={"projection": proj}
        )
        panel_names = ["targets","predictions","diff"]
        titles = ["Targets","Predictions","Diff"]

        for ax, ttl in zip(axes, titles):
            ax.set_title(ttl)
            ax.set_global()
            ax.set_extent([0, 360, -90, 90], crs=proj)
            ax.set_xticks([]); ax.set_yticks([])

        # Targets
        im0 = axes[0].imshow(tgt_plot.sel(step=stp), origin="lower", extent=extent,
                             vmin=vmin_main, vmax=vmax_main, transform=proj)
        c0 = fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.02); c0.set_label(units or "")

        # Predictions
        im1 = axes[1].imshow(pred_plot.sel(step=stp), origin="lower", extent=extent,
                             vmin=vmin_main, vmax=vmax_main, transform=proj)
        c1 = fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.02); c1.set_label(units or "")

        # Diff
        im2 = axes[2].imshow(diff.sel(step=stp), origin="lower", extent=extent,
                             cmap="RdBu_r", vmin=vmin_diff, vmax=vmax_diff, transform=proj)
        c2 = fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.02)
        c2.set_label(f"{aifs_var} difference ({units})" if units else f"{aifs_var} difference")

        # Coastlines & borders where requested
        for ax, name in zip(axes, panel_names):
            if name in add_coastlines:
                ax.coastlines(resolution="110m", linewidth=0.5)
                ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=":")

        fig.suptitle(f"{nice_name(aifs_var)}, {pd.to_timedelta(stp,'h')}", y=1.02, fontsize=14)
        return fig

    # Controls
    play    = widgets.Play(interval=400, value=0, min=0, max=len(steps)-1, step=1)
    pause   = widgets.Button(icon="stop")
    b_first = widgets.Button(icon="step-backward")
    b_prev  = widgets.Button(icon="backward")
    b_next  = widgets.Button(icon="forward")
    b_last  = widgets.Button(icon="step-forward")
    loop    = widgets.ToggleButtons(options=["Once", "Loop", "Reflect"], value="Loop")
    slider  = widgets.IntSlider(value=0, min=0, max=len(steps)-1, step=1, readout=False)
    widgets.jsdlink((play, "value"), (slider, "value"))

    def render(i):
        with out:
            out.clear_output(wait=True)
            fig = draw(i)
            display(fig)
            plt.close(fig)

    def on_slider(change):
        if change["name"] == "value":
            render(change["new"])

    def on_first(_): slider.value = slider.min
    def on_prev(_):  slider.value = max(slider.min, slider.value - 1)
    def on_next(_):  slider.value = min(slider.max, slider.value + 1)
    def on_last(_):  slider.value = slider.max
    def on_pause(_): play._playing = False

    def while_playing(change):
        if change["name"] != "value": return
        if change["new"] == slider.max:
            if loop.value == "Once":
                play._playing = False
            elif loop.value == "Loop":
                slider.value = slider.min
            elif loop.value == "Reflect":
                play.step = -abs(play.step)
        elif change["new"] == slider.min and loop.value == "Reflect":
            play.step = abs(play.step)

    render(0)
    slider.observe(on_slider, names="value")
    b_first.on_click(on_first); b_prev.on_click(on_prev)
    b_next.on_click(on_next);   b_last.on_click(on_last)
    pause.on_click(on_pause)
    play.observe(while_playing, names="value")

    controls = widgets.HBox([play, pause, b_first, b_prev, b_next, b_last, loop])
    display(out); display(slider); display(controls)


In [9]:
make_triptych_widgets_static_refresh(output_ds, FULL_ERA5, aifs_var="2t")
make_triptych_widgets_static_refresh(output_ds, FULL_ERA5, aifs_var="tp")
make_triptych_widgets_static_refresh(output_ds, FULL_ERA5, aifs_var="z_500")

: 

In [31]:

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

import cartopy.crs as ccrs
import cartopy.feature as cfeature

VAR_MAP = {
    "2t":    "2m_temperature",      # Kelvin
    "z_500": "geopotential",        # m^2 s^-2 (convert to meters)
    "tp":    "total_precipitation", # meters (convert to mm)
}
G = 9.80665  # m s^-2

def _to_celsius(da: xr.DataArray) -> xr.DataArray:
    return (da - 273.15).assign_attrs(units="°C")

def _era5_align_to_aifs(da: xr.DataArray, aifs_like: xr.DataArray) -> xr.DataArray:
    ren = {}
    if "latitude" in da.dims:  ren["latitude"]  = "lat"
    if "longitude" in da.dims: ren["longitude"] = "lon"
    if ren: da = da.rename(ren)
    if float(da.lon.min()) < 0 or float(da.lon.max()) <= 180:
        da = da.assign_coords(lon=(da.lon % 360))
    if not np.all(np.diff(da.lon.values) > 0): da = da.sortby("lon")
    if not np.all(np.diff(da.lat.values) > 0): da = da.sortby("lat")
    out = da.interp(lat=aifs_like.lat.values, lon=aifs_like.lon.values)
    return out.sortby(["lat","lon"])

def make_triptych_widgets_static_refresh(
    output_ds: xr.Dataset,
    era5: xr.Dataset,
    aifs_var: str = "2t",
    vname_map: dict = VAR_MAP,
    use_percentile_limits: bool = True,
    diff_clip_pct: float = 99.5,
    add_coastlines=("targets","predictions"),  # <- which panels get coastlines
):
    if aifs_var not in output_ds:
        raise KeyError(f'"{aifs_var}" not in output_ds. Include it in save_vars before running inference.')
    if aifs_var not in vname_map:
        raise KeyError(f'No ERA5 mapping for "{aifs_var}". Add it to VAR_MAP.')

    init_time = pd.to_datetime(output_ds.time.values[0])
    steps = output_ds.step.values.astype(int)  # 6,12,...,360
    valid_times = [init_time + pd.to_timedelta(int(h), "h") for h in steps]

    pred = output_ds[aifs_var].isel(time=0).sortby(["lat","lon"])  # (step, lat, lon)

    era5_var = vname_map[aifs_var]
    if aifs_var == "z_500":
        tgt_raw = era5[era5_var].sel(time=valid_times, level=500, method="nearest") / G
        tgt_raw = tgt_raw.rename("z_500")
    else:
        tgt_raw = era5[era5_var].sel(time=valid_times, method="nearest")
    tgt = _era5_align_to_aifs(tgt_raw, pred).assign_coords(step=("time", steps)).swap_dims(time="step")

    # conversions
    if aifs_var == "2t":
        pred_plot = _to_celsius(pred);  tgt_plot = _to_celsius(tgt);  units = "°C"
    elif aifs_var == "z_500":
        pred_plot = (pred / G).assign_attrs(units="m");  tgt_plot = tgt.assign_attrs(units="m");  units = "m"
    elif aifs_var == "tp":
        pred_plot = (pred * 1000.0).assign_attrs(units="mm");  tgt_plot = (tgt * 1000.0).assign_attrs(units="mm");  units = "mm"
    else:
        pred_plot = pred; tgt_plot = tgt; units = pred.attrs.get("units","")

    diff = (pred_plot - tgt_plot).astype(np.float32)

    # absolute-panel limits
    if use_percentile_limits:
        pair = np.concatenate([pred_plot.values.ravel(), tgt_plot.values.ravel()])
        vmin_main = float(np.nanpercentile(pair, 0.5))
        vmax_main = float(np.nanpercentile(pair, 99.5))
    else:
        vmin_main = float(min(pred_plot.min().item(), tgt_plot.min().item()))
        vmax_main = float(max(pred_plot.max().item(), tgt_plot.max().item()))

    # diff limits (global)
    a_global = float(np.nanpercentile(np.abs(diff.values.ravel()), diff_clip_pct)) if use_percentile_limits \
               else float(np.nanmax(np.abs(diff.values)))

    extent = [float(pred_plot.lon.min()), float(pred_plot.lon.max()),
              float(pred_plot.lat.min()), float(pred_plot.lat.max())]
    out = widgets.Output()

    def nice_name(var):
        return {"2t":"2m_temperature", "z_500":"z500 (m)", "tp":"total_precipitation (mm)"} \
               .get(var, var)

    def draw(i: int):
        stp = int(steps[i])
        vmin_diff, vmax_diff = -a_global, a_global

        # Cartopy GeoAxes for coastlines
        proj = ccrs.PlateCarree()
        fig, axes = plt.subplots(
            1, 3, figsize=(18, 4), constrained_layout=True,
            subplot_kw={"projection": proj}
        )
        panel_names = ["targets","predictions","diff"]
        titles = ["Targets","Predictions","Diff"]

        for ax, ttl in zip(axes, titles):
            ax.set_title(ttl)
            ax.set_global()
            ax.set_extent([0, 360, -90, 90], crs=proj)
            ax.set_xticks([]); ax.set_yticks([])

        # Targets
        im0 = axes[0].imshow(tgt_plot.sel(step=stp), origin="lower", extent=extent,
                             vmin=vmin_main, vmax=vmax_main, transform=proj)
        c0 = fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.02); c0.set_label(units or "")

        # Predictions
        im1 = axes[1].imshow(pred_plot.sel(step=stp), origin="lower", extent=extent,
                             vmin=vmin_main, vmax=vmax_main, transform=proj)
        c1 = fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.02); c1.set_label(units or "")

        # Diff
        im2 = axes[2].imshow(diff.sel(step=stp), origin="lower", extent=extent,
                             cmap="RdBu_r", vmin=vmin_diff, vmax=vmax_diff, transform=proj)
        c2 = fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.02)
        c2.set_label(f"{aifs_var} difference ({units})" if units else f"{aifs_var} difference")

        # Coastlines & borders where requested
        for ax, name in zip(axes, panel_names):
            if name in add_coastlines:
                ax.coastlines(resolution="110m", linewidth=0.5)
                ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=":")

        fig.suptitle(f"{nice_name(aifs_var)}, {pd.to_timedelta(stp,'h')}", y=1.02, fontsize=14)
        return fig

    # Controls
    play    = widgets.Play(interval=400, value=0, min=0, max=len(steps)-1, step=1)
    pause   = widgets.Button(icon="stop")
    b_first = widgets.Button(icon="step-backward")
    b_prev  = widgets.Button(icon="backward")
    b_next  = widgets.Button(icon="forward")
    b_last  = widgets.Button(icon="step-forward")
    loop    = widgets.ToggleButtons(options=["Once", "Loop", "Reflect"], value="Loop")
    slider  = widgets.IntSlider(value=0, min=0, max=len(steps)-1, step=1, readout=False)
    widgets.jsdlink((play, "value"), (slider, "value"))

    def render(i):
        with out:
            out.clear_output(wait=True)
            fig = draw(i)
            display(fig)
            plt.close(fig)

    def on_slider(change):
        if change["name"] == "value":
            render(change["new"])

    def on_first(_): slider.value = slider.min
    def on_prev(_):  slider.value = max(slider.min, slider.value - 1)
    def on_next(_):  slider.value = min(slider.max, slider.value + 1)
    def on_last(_):  slider.value = slider.max
    def on_pause(_): play._playing = False

    def while_playing(change):
        if change["name"] != "value": return
        if change["new"] == slider.max:
            if loop.value == "Once":
                play._playing = False
            elif loop.value == "Loop":
                slider.value = slider.min
            elif loop.value == "Reflect":
                play.step = -abs(play.step)
        elif change["new"] == slider.min and loop.value == "Reflect":
            play.step = abs(play.step)

    render(0)
    slider.observe(on_slider, names="value")
    b_first.on_click(on_first); b_prev.on_click(on_prev)
    b_next.on_click(on_next);   b_last.on_click(on_last)
    pause.on_click(on_pause)
    play.observe(while_playing, names="value")

    controls = widgets.HBox([play, pause, b_first, b_prev, b_next, b_last, loop])
    display(out); display(slider); display(controls)


In [32]:
make_triptych_widgets_static_refresh(output_ds, FULL_ERA5, aifs_var="2t",add_coastlines=("targets","predictions","diff"))
make_triptych_widgets_static_refresh(output_ds, FULL_ERA5, aifs_var="tp",add_coastlines=("targets","predictions","diff"))
make_triptych_widgets_static_refresh(output_ds, FULL_ERA5, aifs_var="z_500",add_coastlines=("targets","predictions","diff"))

: 